In [2]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 0.13
# ===================================================================
# AUTOR: Carmen Gomez Alarcon
# FECHA: 2026
# FASE: Paso 6. Generación de gráficos para resultados de CLASIFICACIÓN
# DESCRIPCION: Generacion de graficos a partir de resultados guardados
# ===================================================================

# Importar librerias necesarias
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurar matplotlib para guardar imagenes sin ventana emergente
import matplotlib
matplotlib.use('Agg')

print("="*60)
print("STEP 6: GENERATING PLOTS FROM SAVED RESULTS")
print("="*60)

# Directorios donde estan los resultados
INPUT_DIR = 'ml_results_classification'
OUTPUT_DIR = 'ml_results_classification'

# ==========================================
# 1. CARGAR RESULTADOS GUARDADOS
# ==========================================

# Cargar diccionario con metricas de todos los modelos
with open(os.path.join(INPUT_DIR, 'results.pkl'), 'rb') as f:
    results = pickle.load(f)

# Cargar diccionario con predicciones de todos los modelos
with open(os.path.join(INPUT_DIR, 'predictions.pkl'), 'rb') as f:
    predictions = pickle.load(f)

# Cargar etiquetas reales del conjunto de prueba
y_test = np.load(os.path.join(INPUT_DIR, 'y_test.npy'))

# Cargar matriz de confusion (cuantos aciertos y errores por clase)
cm = np.load(os.path.join(INPUT_DIR, 'confusion_matrix.npy'))

# Cargar resumen de todos los modelos (accuracy, f1, etc.)
df_summary = pd.read_excel(os.path.join(INPUT_DIR, 'models_summary.xlsx'))
df_summary = df_summary.set_index(df_summary.columns[0])  # La primera columna es el nombre del modelo

# Intentar cargar importancia de caracteristicas (si existe)
feature_importance_path = os.path.join(INPUT_DIR, 'feature_importance.npy')
selected_features_path = os.path.join(INPUT_DIR, '..', 'ml_normalized_grouped', 'selected_features.pkl')

if os.path.exists(feature_importance_path) and os.path.exists(selected_features_path):
    importances = np.load(feature_importance_path)  # Valores de importancia
    with open(selected_features_path, 'rb') as f:
        selected_features = pickle.load(f)  # Nombres de las caracteristicas
else:
    importances = None
    selected_features = None

# ==========================================
# 1b. NOMBRES DE CLASES (ACTUALIZADO A 3 CATEGORÍAS)
# ==========================================
class_names = ['Empty(0)', 'Low(1-30)', 'High(31+)']  # <--- CAMBIO CLAVE

# Identificar el mejor modelo (el primero en el resumen, ordenado por F1)
best = df_summary.index[0] if len(df_summary) > 0 else "Random Forest"

print("   Loaded results for", len(results), "models")
print("   Best model:", best)

# ==========================================
# 2. GRÁFICO DE BARRAS - COMPARATIVA DE TODOS LOS MODELOS
# ==========================================
# Este gráfico muestra TODAS las métricas de TODOS los modelos en un solo gráfico

fig, ax = plt.subplots(figsize=(12, 7))

# Métricas a comparar (columnas del DataFrame)
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'CV_F1']

# Posiciones de las barras
x = np.arange(len(df_summary.index))  # Posiciones para cada modelo
width = 0.15  # Ancho de cada barra
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']  # Colores para cada métrica

# Crear barras para cada métrica
for i, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    values = df_summary[metric].values if metric in df_summary.columns else [0]*len(df_summary)
    offset = (i - len(metrics_to_plot)/2) * width + width/2
    bars = ax.bar(x + offset, values, width, label=metric, color=color, edgecolor='black', linewidth=0.5)
    
    # Añadir valores encima de las barras (solo para valores > 0)
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                   f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

# Configurar el gráfico
ax.set_xlabel('Models', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('COMPARISON OF ALL MODELS - All Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_summary.index, fontsize=10, rotation=15, ha='right')
ax.set_ylim(0, 1.05)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0.7, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Good threshold (0.7)')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'all_models_comparison.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✅ Saved: all_models_comparison.png")

# ==========================================
# 3. GRÁFICO DE RADAR - PERFIL COMPLETO DE CADA MODELO
# ==========================================
# Muestra el "perfil" de cada modelo en forma de radar

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': 'polar'})

# Métricas para el radar
radar_metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'CV_F1']
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]  # Cerrar el círculo

# Colores para cada modelo
model_colors = plt.cm.tab10(np.linspace(0, 1, len(df_summary.index)))

for idx, (model_name, color) in enumerate(zip(df_summary.index, model_colors)):
    values = []
    for metric in radar_metrics:
        val = df_summary.loc[model_name, metric] if metric in df_summary.columns else 0
        values.append(val)
    values += values[:1]  # Cerrar el círculo
    
    # Dibujar línea para el modelo
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=model_name)
    ax.fill(angles, values, alpha=0.1, color=color)

# Configurar el gráfico de radar
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)
ax.set_title('MODEL RADAR COMPARISON - Complete Profile', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=9)
ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'radar_comparison.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✅ Saved: radar_comparison.png")

# ==========================================
# 4. GRÁFICO DE CALOR - MATRIZ DE MÉTRICAS POR MODELO
# ==========================================
# Mapa de calor que muestra todas las métricas de todos los modelos

fig, ax = plt.subplots(figsize=(10, 6))

# Crear matriz para el heatmap (modelos x métricas)
heatmap_data = df_summary[['Accuracy', 'Precision', 'Recall', 'F1', 'CV_F1']]

sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', 
            xticklabels=True, yticklabels=True,
            linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title('HEATMAP: Metrics by Model', fontsize=14, fontweight='bold')
ax.set_ylabel('Models', fontsize=12, fontweight='bold')
ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'metrics_heatmap.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✅ Saved: metrics_heatmap.png")

# ==========================================
# 5. MATRIZ DE CONFUSION (solo mejor modelo)
# ==========================================
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={"size": 14})
ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual', fontsize=12, fontweight='bold')
ax.set_title(f'Confusion Matrix - {best}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✅ Saved: confusion_matrix.png")

# ==========================================
# 6. IMPORTANCIA DE CARACTERISTICAS
# ==========================================
if importances is not None and selected_features is not None:
    indices = np.argsort(importances)[::-1]
    n_top = min(15, len(selected_features))
    
    fig, ax = plt.subplots(figsize=(10, 6))
    top_feat = [selected_features[indices[i]] for i in range(n_top)]
    top_imp = importances[indices[:n_top]]
    
    ax.barh(range(n_top), top_imp[::-1], color='steelblue', edgecolor='navy')
    ax.set_yticks(range(n_top))
    ax.set_yticklabels(top_feat[::-1], fontsize=10)
    ax.set_xlabel('Importance', fontsize=12, fontweight='bold')
    ax.set_title('Feature Importance - Random Forest', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✅ Saved: feature_importance.png")

# ==========================================
# 7. DISTRIBUCION DE CLASES (ACTUALIZADO)
# ==========================================
unique, counts = np.unique(y_test, return_counts=True)
fig, ax = plt.subplots(figsize=(8, 5))
# Colores para 3 clases
colors = ['#2ecc71', '#3498db', '#e74c3c']  # verde, azul, rojo
bars = ax.bar(class_names, counts, color=colors)
ax.set_xlabel('Occupation Level', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Samples', fontsize=12, fontweight='bold')
ax.set_title('Test Set Class Distribution', fontsize=13, fontweight='bold')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            str(count), ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✅ Saved: class_distribution.png")

# ==========================================
# 8. MAPA DE CORRELACION DE METRICAS
# ==========================================
df_metrics = pd.DataFrame(results).T
metrics_corr = df_metrics[['Accuracy', 'Precision', 'Recall', 'F1', 'CV_F1']].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(metrics_corr, annot=True, fmt='.3f', cmap='coolwarm', 
            square=True, ax=ax, annot_kws={"size": 10})
ax.set_title('Metrics Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'metrics_correlation.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✅ Saved: metrics_correlation.png")

# ==========================================
# 9. TABLA DE RESULTADOS (texto)
# ==========================================
print("\n" + "="*60)
print("FULL MODEL COMPARISON TABLE")
print("="*60)
print(df_summary[['Accuracy', 'Precision', 'Recall', 'F1', 'CV_F1']].to_string())
print("="*60)

# ==========================================
# 10. RESUMEN FINAL
# ==========================================
print("\n" + "="*60)
print("PLOTS GENERATED SUCCESSFULLY")
print("="*60)
print(f"Files saved in: {OUTPUT_DIR}/")
print("\n📊 Generated plots:")
print("   1. all_models_comparison.png  - Bar chart with ALL models and ALL metrics")
print("   2. radar_comparison.png       - Radar chart showing complete profile per model")
print("   3. metrics_heatmap.png        - Heatmap of all metrics by model")
print("   4. confusion_matrix.png       - Confusion matrix (best model)")
print("   5. feature_importance.png     - Feature importance (if available)")
print("   6. class_distribution.png     - Distribution of test classes")
print("   7. metrics_correlation.png    - Correlation between metrics")
print("="*60)

STEP 6: GENERATING PLOTS FROM SAVED RESULTS
   Loaded results for 7 models
   Best model: Random Forest
   ✅ Saved: all_models_comparison.png
   ✅ Saved: radar_comparison.png
   ✅ Saved: metrics_heatmap.png
   ✅ Saved: confusion_matrix.png
   ✅ Saved: feature_importance.png
   ✅ Saved: class_distribution.png
   ✅ Saved: metrics_correlation.png

FULL MODEL COMPARISON TABLE
                     Accuracy  Precision  Recall      F1     CV_F1
Unnamed: 0                                                        
Random Forest            0.84     0.8498    0.84  0.8394  0.845245
LightGBM                 0.80     0.7870    0.80  0.7852  0.376684
Logistic Regression      0.76     0.8111    0.76  0.7589  0.888424
XGBoost                  0.76     0.7487    0.76  0.7490  0.889528
Gradient Boosting        0.72     0.7261    0.72  0.7095  0.861213
SVM                      0.68     0.7314    0.68  0.6726  0.888446
Decision Tree            0.68     0.6687    0.68  0.6638  0.780902

PLOTS GENERATED SUCCE